# Predictive Process Monitoring — Suffix Prediction Benchmark

This notebook provides a **single entry point** to reproduce all experiments from the paper:

> *SuTraN: an Encoder-Decoder Transformer for Full-Context-Aware Suffix Prediction of Business Processes*  
> ICPM 2024

It also includes the **BEST** baseline introduced in:

> *BEST: Bilaterally Expanding Subtrace Tree for Event Sequence Prediction*  
> Rauch et al., BPM 2025

---

## What this notebook does

1. **Creates pre-processed tensor datasets** from the raw event logs (BPIC17, BPIC17-DR, BPIC19).
2. **Trains and evaluates every model** on each event log and saves results to disk.

Each model section includes a short description, the paper it is based on, and a code cell that runs `train_eval()` for every log.

---

## Models covered

| # | Model | Type | Paper |
|---|-------|------|-------|
| 1 | **SuTraN (DA)** | Encoder-Decoder Transformer | SuTraN paper (ICPM 2024) |
| 2 | **SuTraN (NDA)** | Encoder-Decoder Transformer (control-flow only) | SuTraN paper (ICPM 2024) |
| 3 | **CRTP-LSTM (DA)** | Multi-task LSTM | Camargo et al. (2019) |
| 4 | **CRTP-LSTM (NDA)** | Multi-task LSTM (control-flow only) | Camargo et al. (2019) |
| 5 | **ED-LSTM** | Encoder-Decoder LSTM (seq2seq) | Tax et al. (2017) / Pfeiffer et al. (2021) |
| 6 | **SEP-LSTM** | One-step-ahead LSTM with iterative feedback | Taymouri et al. (2021) |
| 7 | **BEST** | N-gram pattern tree (no neural network) | Rauch et al., BPM 2025 |

**DA** = Data-Aware (uses all event / case attributes)  
**NDA** = Non-Data-Aware (uses only activity labels and timestamp proxies)

---

## Event logs

| Log | Cases | Mean length | Raw file |
|-----|-------|-------------|----------|
| BPIC17 | 30,078 | 36.90 | `bpic17_with_loops.csv` |
| BPIC17-DR | 30,078 | 23.42 | `BPIC17_no_loop.csv` |
| BPIC19 | 181,395 | 5.44 | `BPIC19.csv` |

Place the CSV files in the repository root before running the dataset creation cells.

In [1]:
# ── Shared configuration ────────────────────────────────────────────────────
# Log names must match the log_name used in log_to_tensors() (set inside each
# create_*_data.py script).  tss_index is the zero-based position of the
# 'time since start' column inside the numerical prefix feature tensor;
# it equals the number of numerical event + case features for that log.

LOGS = [
    #{"log_name": "BPIC_17",    "tss_index": 5},  # 4 num. event fts + 1 num. case ft
    #{"log_name": "BPIC_17_DR", "tss_index": 5},  # same numerical features, no loops
    #{"log_name": "BPIC_19",    "tss_index": 1},  # 1 num. event ft, no num. case fts
    {"log_name": "Sepsis",    "tss_index": 4},  
]

print("Logs configured:", [l['log_name'] for l in LOGS])

Logs configured: ['Sepsis']


---
# Part 1 — Creating Datasets

Each script reads a raw CSV event log, applies log-specific preprocessing (timestamp parsing, invalid-case removal, feature engineering), then calls `log_to_tensors()` from `Preprocessing/from_log_to_tensors.py`.

The pipeline:
1. Filters cases by date range and maximum duration.
2. Maps categorical features to integers.
3. Standardises numerical features using training-set statistics.
4. Creates all prefix–suffix pairs (sliding window).
5. Splits into train / validation / test using an **out-of-time** split with data-leakage prevention.
6. Saves `train_tensordataset.pt`, `val_tensordataset.pt`, `test_tensordataset.pt` and metadata pickles inside a subfolder named after the log.

> **Note:** Run each cell only once. Re-running will overwrite the saved tensors.

In [4]:
from create_BPIC17_OG_data import construct_BPIC17_datasets

construct_BPIC17_datasets()

Generating Dataframes...


/app/Preprocessing/create_benchmarks.py:85: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  case_stops_df['date'] = case_stops_df[timestamp].dt.to_period('M')
100%|█████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 50.57it/s]


Generating Tensors...
Computing train set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing validation set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing test set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...


In [2]:
from create_BPIC17_DR_data import construct_BPIC17_DR_datasets

construct_BPIC17_DR_datasets()

Generating Dataframes...


/app/Preprocessing/create_benchmarks.py:85: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  case_stops_df['date'] = case_stops_df[timestamp].dt.to_period('M')
100%|█████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 69.61it/s]


Generating Tensors...
Computing train set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing validation set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing test set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...


In [2]:
from create_BPIC19_data import construct_BPIC19_datasets

construct_BPIC19_datasets()

Generating Dataframes...


/app/Preprocessing/create_benchmarks.py:58: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  case_starts_df['date'] = case_starts_df[timestamp].dt.to_period('M')
/app/Preprocessing/create_benchmarks.py:85: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  case_stops_df['date'] = case_stops_df[timestamp].dt.to_period('M')
/app/Preprocessing/create_benchmarks.py:115: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  case_starts_df['date'] = case_starts_df[timestamp].dt.to_period('M')
100%|███████████████████████████████████████████████████████████████| 13/13 [00:00<00:00, 55.41it/s]


Generating Tensors...
Computing train set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing validation set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing test set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...


# Any dataset

In [2]:
from create_general_data import construct_datasets

# ------------------------------------------------------------------ #
# Edit the variables below to match your event log.                   #
# ------------------------------------------------------------------ #

LOG_PATH   = 'Logs/Sepsis.xes.gz'   # or 'my_log.csv'
LOG_NAME   = 'Sepsis'

# Standard XES column names — change if your CSV uses different names.
CASE_ID    = 'case:concept:name'
ACT_LABEL  = 'concept:name'
TIMESTAMP  = 'time:timestamp'

# Set to None to auto-detect from column prefixes / dtypes, or provide
# explicit lists as in the log-specific create_*.py files.
CAT_CASEFTS  = None
NUM_CASEFTS  = None
CAT_EVENTFTS = None
NUM_EVENTFTS = None

# Filtering / splitting parameters — adjust to your log's date range
# and case-length distribution.
START_DATE        = None   # e.g. "2018-01"
START_BEFORE_DATE = None   # e.g. "2018-09"
END_DATE          = None   # e.g. "2019-02"
MAX_DAYS          = None   # e.g. 143.33
WINDOW_SIZE       = None   # None → auto (98.5th percentile)
TEST_LEN_SHARE    = 0.20
VAL_LEN_SHARE     = 0.20
MODE              = 'preferred' # preferred or workaround
OUTCOME           = None
PLOT              = True    # set to False to skip the split visualisation

construct_datasets(
    log_path=LOG_PATH,
    log_name=LOG_NAME,
    case_id=CASE_ID,
    act_label=ACT_LABEL,
    timestamp=TIMESTAMP,
    cat_casefts=CAT_CASEFTS,
    num_casefts=NUM_CASEFTS,
    cat_eventfts=CAT_EVENTFTS,
    num_eventfts=NUM_EVENTFTS,
    outcome=OUTCOME,
    start_date=START_DATE,
    start_before_date=START_BEFORE_DATE,
    end_date=END_DATE,
    max_days=MAX_DAYS,
    window_size=WINDOW_SIZE,
    test_len_share=TEST_LEN_SHARE,
    val_len_share=VAL_LEN_SHARE,
    mode=MODE,
    plot=PLOT,
)

parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

Split plot saved to 'results_per_log/Sepsis/Sepsis_preferred_split.png'
Auto-detected features:
  cat_casefts  : []
  num_casefts  : []
  cat_eventfts : ['InfectionSuspected', 'org:group', 'DiagnosticBlood', 'DisfuncOrg', 'SIRSCritTachypnea', 'Hypotensie', 'SIRSCritHeartRate', 'Infusion', 'DiagnosticArtAstrup', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther', 'SIRSCriteria2OrMore', 'DiagnosticXthorax', 'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 'Oligurie', 'DiagnosticLacticAcid', 'lifecycle:transition', 'Diagnose', 'Hypoxie', 'DiagnosticUrinarySediment', 'DiagnosticECG']
  num_eventfts : ['Age', 'Leucocytes', 'CRP', 'LacticAcid']
Auto-derived window_size (98.5th percentile): 41
Auto-derived max_days (maximum case duration): 422.32
Generating Dataframes...


100%|██████████████████████████████████████████| 26/26 [00:00<00:00, 625.41it/s]


Cases – train: 599  val: 150  test: 285
Pairs – train: 8025  val: 1904  test: 4057
Generating Tensors...
Computing train set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing validation set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing test set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
Tensors saved to 'results_per_log/Sepsis/'
tss_index = 4  (= 0 num_casefts + 4 num_eventfts)
Use this value for the tss_index parameter in TRAIN_EVAL_*.py scripts.


---
# Part 2 — Model Training & Evaluation

Each section below trains one model on every event log and automatically evaluates it on the held-out test set.  Results (DL similarity, TTNE MAE, RRT MAE, per-length dictionaries) are saved under `<log_name>/<model>_results/TEST_SET_RESULTS/`.

Training is GPU-accelerated when a CUDA device is available.  Models use early stopping based on combined validation ranking of DL similarity and RRT MAE.

---
## Model 1 — SuTraN (Data-Aware)

**Approach from paper:** *SuTraN: an Encoder-Decoder Transformer for Full-Context-Aware Suffix Prediction of Business Processes* (ICPM 2024)

SuTraN is an **encoder-decoder Transformer** designed specifically for multi-task suffix prediction in PPM.  Key properties:

- The **encoder** is a stack of self-attention layers that builds a full contextual representation of the observed prefix, using *all* available event and case attributes (Data-Aware).
- The **decoder** generates the complete activity and timestamp suffix in a **single forward pass** during training (teacher forcing), and autoregressively at inference.
- **Cross-attention** at every decoder step lets each predicted suffix token attend to the entire encoded prefix — giving the model its *full-context-aware* property.
- **Multi-task heads** simultaneously predict: activity labels, time-till-next-event (TTNE) at each suffix step, and a direct remaining runtime (RRT) estimate.
- Activity embeddings are **shared** between encoder and decoder.

Architecture defaults: `d_model=32`, 4 encoder + 4 decoder layers, 8 attention heads, `d_ff=128`, dropout 0.2, AdamW + exponential LR decay, up to 200 epochs with patience 24.

In [ ]:
import TRAIN_EVAL_SUTRAN_DA as sutran_da

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"SuTraN (DA) — {cfg['log_name']}")
    print(f"{'='*60}")
    sutran_da.train_eval(log_name=cfg["log_name"])


SuTraN (DA) — Sepsis
18
device: cpu


/workspace/baselines/SuffixTransformerNetwork/TRAIN_EVAL_SUTRAN_DA.py:172: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_dataset = torch.load(temp_path)
/workspace/bas

Device: cpu
 
------------------------------------
EPOCH 0:
____________________________________


Batch calculation at epoch 0.: 63it [00:14,  4.45it/s]


End of epoch 0
Running average global loss: 0.2603326544165611 (over last 800 batches)
Running average activity prediction loss: 0.19019298911094665 (Cross Entropy over last 800 batches)
Running average time till next event prediction loss: 0.025191943403333427 (MAE over last 800 batches)
Running average (complete) remaining runtime prediction loss: 0.044947720989584924 (MAE over last 800 batches)


...

Validation batch calculation: 1it [00:15, 15.19s/it]


Avg MAE TTNE prediction validation set: 0.062409061938524246 (standardized) ; 1360.7404947916666 (minutes)'
Avg 1-(normalized) DL distance acitivty suffix prediction validation set: 0.35260477662086487
Percentage of suffixes predicted to END: too early - 0.1171218487394958 ; right moment - 0.12867647058823528 ; too late - 0.7542016806722689
Too early instances - avg amount of events too early: 4.1569504737854
Too late instances - avg amount of events too late: 24.19498634338379
Avg absolute amount of events predicted too early / too late: 18.734769821166992
Avg MAE RRT prediction validation set: 0.12428104877471924 (standardized) ; 7705.383854166666 (minutes)'
 
------------------------------------
EPOCH 38:
____________________________________


Batch calculation at epoch 38.: 63it [00:15,  4.17it/s]


End of epoch 38
Running average global loss: 0.13801247864961624 (over last 800 batches)
Running average activity prediction loss: 0.0894137080013752 (Cross Entropy over last 800 batches)
Running average time till next event prediction loss: 0.013462701737880707 (MAE over last 800 batches)
Running average (complete) remaining runtime prediction loss: 0.035136067867279054 (MAE over last 800 batches)


Validation batch calculation: 0it [00:00, ?it/s]

---
## Model 2 — SuTraN (Non-Data-Aware)

**Approach from paper:** *SuTraN* (ICPM 2024) — NDA variant

Identical architecture to SuTraN (DA) above, but **restricted to control-flow information only**: each prefix event token consists solely of the activity label and the two numerical timestamp proxies (time since case start, time since previous event).  All additional categorical and numerical event/case attributes are discarded.

This variant is included to provide a fair comparison against the other NDA baselines (CRTP-LSTM NDA, ED-LSTM, SEP-LSTM, BEST) that also operate without data attributes.

In [ ]:
import TRAIN_EVAL_SUTRAN_NDA as sutran_nda

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"SuTraN (NDA) — {cfg['log_name']}")
    print(f"{'='*60}")
    sutran_nda.train_eval(log_name=cfg["log_name"], tss_index=cfg["tss_index"])

---
## Model 3 — CRTP-LSTM (Data-Aware)

**Approach from paper:** *Camargo, M., Dumas, M., González-Rojas, O. (2019). Learning Accurate LSTM Models of Business Processes. BPM 2019.*

CRTP-LSTM (Complete Remaining Trace Prediction LSTM) is a **multi-task LSTM** baseline:

- A **shared LSTM** encoder processes the prefix and feeds into multiple **dedicated LSTM** heads, one per prediction target.
- Prediction targets: activity label suffix, TTNE suffix, and RRT (remaining runtime).
- The suffix is generated by the LSTM in a single recurrent pass over the decoder input (teacher-forced during training, autoregressive at inference).
- Prefix events are **left-padded** so the final hidden state always corresponds to the last observed event.
- The DA variant uses all available event and case attributes.

Architecture defaults: `d_model=80`, 1 shared + 1 dedicated LSTM layer, dropout 0.2, NAdam optimiser with ReduceLROnPlateau, up to 500 epochs.

In [ ]:
import TRAIN_EVAL_CRTP_LSTM_DA as crtp_da

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"CRTP-LSTM (DA) — {cfg['log_name']}")
    print(f"{'='*60}")
    crtp_da.train_eval(log_name=cfg["log_name"])

---
## Model 4 — CRTP-LSTM (Non-Data-Aware)

**Approach from paper:** *Camargo et al. (2019)* — NDA variant

Same CRTP-LSTM architecture as Model 3, but restricted to the activity label and two timestamp proxies per event (no additional attributes).  Provides a fair NDA comparison partner to SuTraN NDA.

In [ ]:
import TRAIN_EVAL_CRTP_LSTM_ND as crtp_nda

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"CRTP-LSTM (NDA) — {cfg['log_name']}")
    print(f"{'='*60}")
    crtp_nda.train_eval(log_name=cfg["log_name"], tss_index=cfg["tss_index"])

---
## Model 5 — ED-LSTM (Encoder-Decoder LSTM)

**Approach based on:**  
- *Tax, N. et al. (2017). Predictive Business Process Monitoring with LSTM Neural Networks. CAiSE 2017.*  
- *Pfeiffer, P. et al. (2021). seq2seq Encoder-Decoder for Suffix Prediction. EDOC 2021.*

ED-LSTM is a **sequence-to-sequence encoder-decoder LSTM**:

- The **encoder LSTM** reads the full prefix and produces a fixed-size context vector (final hidden + cell state).
- The **decoder LSTM** is initialised with the encoder's final state and generates the activity and TTNE suffix step-by-step.
- Compared to the CRTP-LSTM, the decoder has its own recurrent state that is updated at each suffix step, which allows it to use previously predicted outputs as context.
- NDA only (activity label + timestamp proxies).

In [ ]:
import TRAIN_EVAL_ED_LSTM as ed_lstm

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"ED-LSTM — {cfg['log_name']}")
    print(f"{'='*60}")
    ed_lstm.train_eval(log_name=cfg["log_name"], tss_index=cfg["tss_index"])

---
## Model 6 — SEP-LSTM (Single-Event-Prediction LSTM)

**Approach from paper:** *Taymouri, F. et al. (2021). Predictive Business Process Monitoring via Generative Adversarial Nets. ICPM 2020 / BPM 2021.*

SEP-LSTM is a **one-step-ahead LSTM** repurposed for suffix generation through an **iterative external feedback loop**:

- The model is trained to predict only the **next** activity and TTNE given the full current prefix.
- At inference, the prediction for step $t$ is appended to the prefix as a new (synthesised) event token, and the model is queried again for step $t+1$.
- This loop continues until the END token is predicted or the maximum suffix length is reached.
- The new event token's timestamp features (time-since-start, time-since-previous) are computed from the predicted TTNE.
- NDA only; uses left-padded prefix tensors.

In [ ]:
import TRAIN_EVAL_SEP_LSTM as sep_lstm

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"SEP-LSTM — {cfg['log_name']}")
    print(f"{'='*60}")
    sep_lstm.train_eval(log_name=cfg["log_name"], tss_index=cfg["tss_index"])

---
## Model 7 — BEST (Bilaterally Expanding Subtrace Tree)

**Approach from paper:** *Rauch, S., Frey, C. M. M., Maldonado, A. J., & Seidl, T. (2025). BEST: Bilaterally Expanding Subtrace Tree for Event Sequence Prediction. BPM 2025. Springer, LNCS 16044.*  
🏆 Runner-up Best Student Paper Award, BPM 2025

BEST is a **non-parametric, data-mining-based** baseline — it requires **no gradient-based optimisation** at all:

- **Fitting:** For every training instance the complete trace (prefix + suffix) is reconstructed. All sub-sequences of length 1 to `max_context_length` (default 10) are extracted and stored as conditional next-activity frequency counts in a dictionary-based n-gram table.
- **Prediction:** Given a test prefix, BEST performs a **longest-context-first back-off search**: it tries to match the last $k$ activities in the current sequence against the table (starting at $k=10$, backing off to $k=1$), and picks the activity with the highest conditional count. Falls back to the global unigram mode if no match is found.
- **Suffix generation:** Activities are generated autoregressively until the END token is predicted.
- **Control-flow only (NDA):** No timestamp or attribute information is used.
- **Time metrics:** Since BEST does not predict timestamps, TTNE and RRT are approximated with a constant predictor equal to the training mean TTNE.

Despite its simplicity, BEST achieves competitive or superior activity-suffix DL similarity compared to deep learning models on several benchmark logs.

In [ ]:
import TRAIN_EVAL_BEST as best

for cfg in LOGS:
    print(f"\n{'='*60}")
    print(f"BEST — {cfg['log_name']}")
    print(f"{'='*60}")
    best.train_eval(log_name=cfg["log_name"])

---
# Part 3 — Results Summary

After all cells above have finished, each model's results are stored as pickle files under:

```
<log_name>/
  SUTRAN_DA_results/TEST_SET_RESULTS/averaged_results.pkl
  SUTRAN_NDA_results/TEST_SET_RESULTS/averaged_results.pkl
  CRTP_LSTM_DA_results/TEST_SET_RESULTS/averaged_results.pkl
  CRTP_LSTM_ND_results/TEST_SET_RESULTS/averaged_results.pkl
  ED_LSTM_results/TEST_SET_RESULTS/averaged_results.pkl
  SEP_LSTM_results/TEST_SET_RESULTS/averaged_results.pkl
  BEST_results/TEST_SET_RESULTS/averaged_results.pkl
```

Each `averaged_results.pkl` is a dict with keys `"DL sim"`, `"MAE TTNE minutes"`, `"MAE RRT minutes"`.

The cell below collects and displays them in a summary table.

In [3]:
from results_collector import get_suffix_baseline_results
import pandas as pd

EVENT_LOGS = [
    "BPIC15_1",
    "BPIC15_2",
    "BPIC15_3",
    "BPIC15_4",
    "BPIC15_5",
    "RequestForPayment",
    "Sepsis",
    "DomesticDeclarations",
    "PrepaidTravelCost",
    "InternationalDeclarations",
    #"BPI_Challenge_2012", # It have been splitted in 3 files A,O,W. Many papers do this.
    "BPI_Challenge_2012_A",
    "BPI_Challenge_2012_O",
    "BPI_Challenge_2012_W",
    "Hospital_Billing",
    "Road_Traffic_Fine_Management_Process",
    "BPI Challenge 2017"
]

df_results = get_suffix_baseline_results(EVENT_LOGS, base_dir="results_per_log")
for log_name, group in df_results.groupby(level="Log"):
    print(f"\n=== Log: {log_name} ===")

    print(
        group.reset_index()[["Model", 'DL similarity ↑ mean', 'MAE TTNE (min) ↓ mean','MAE RRT (min) ↓ mean']]
        #[['Runs', 'DL similarity ↑ mean', 'DL similarity ↑ std',
       #'MAE TTNE (min) ↓ mean', 'MAE TTNE (min) ↓ std', 'MAE RRT (min) ↓ mean',
       #'MAE RRT (min) ↓ std']]
             .sort_values(by="DL similarity ↑ mean", ascending=False)
    )


=== Log: BPI Challenge 2017 ===
             Model  DL similarity ↑ mean  MAE TTNE (min) ↓ mean  \
0      SuTraN (DA)                   NaN                    NaN   
1     SuTraN (NDA)                   NaN                    NaN   
2   CRTP-LSTM (DA)                   NaN                    NaN   
3  CRTP-LSTM (NDA)                   NaN                    NaN   
4          ED-LSTM                   NaN                    NaN   
5         SEP-LSTM                   NaN                    NaN   
6             BEST                   NaN                    NaN   

   MAE RRT (min) ↓ mean  
0                   NaN  
1                   NaN  
2                   NaN  
3                   NaN  
4                   NaN  
5                   NaN  
6                   NaN  

=== Log: BPIC15_1 ===
             Model  DL similarity ↑ mean  MAE TTNE (min) ↓ mean  \
5         SEP-LSTM                0.2132              2424.7000   
1     SuTraN (NDA)                0.2071              2076.9300  